# Conexión Mongo DB

In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['datos_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


# Conectar el Scraper a la BD y guardar
Se usa el script de la semana 3

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "Los AveMayo" #Agegaremos el nombre de grupo dentro del diccionario y la hora en que se obtubo

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("https://www.jumbo.cl/chocolates-galletas-y-snacks/chocolates?nombre_promo=destacados-chocolates-26092025")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz (el que agrupa todos los resultados)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "div.vtex-product-summary-2-x-container"))
        )

        # 2. Captura de Bloques de Primer Nivel (la "tarjeta" o "card" del item)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "div.vtex-product-summary-2-x-container")
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (Selectores específicos)
            # Extraemos el atributo 'title' del enlace para evitar texto truncado
            dato_identificador = bloque.find_element(By.CSS_SELECTOR,"a.vtex-product-summary-2-x-clearLink").get_attribute("title")
            dato_valor = bloque.find_element(By.CLASS_NAME, "span.vtex-product-price-1-x-sellingPriceValue").text
            
            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN: Lógica de Salto de Nodo (Paginación) ---
        try:
            # Buscamos el activador del siguiente nodo de datos
            disparador_siguiente = driver.find_element(By.CSS_SELECTOR, "button.vtex-search-result-3-x-paginationButton--next")
            disparador_siguiente.click()
        except Exception:
            print("Fin del árbol de navegación o nodo no encontrado.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

#TENEMOS EL DICCIONARIO CON UN FORMATO JSON, OBTENIDO DEL SCRIPT. AHORA LO INCLUIMOS EN LA BASE DE DATOS
#Cambios en los datos de salida
from pymongo import MongoClient



# 1. Configuracion de conexion
try:
    # 'database' es el nombre del servicio en el docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Procesamiento e insercion de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza de simbolos y conversion a float
        # Se eliminan simbolos de moneda, comas y espacios en blanco
        valor_sucio = str(dato.get('valor', '0'))
        valor_limpio = valor_sucio.replace('£', '').replace(',', '').strip()
        
        dato['valor'] = float(valor_limpio)
        
        # Insercion individual
        coleccion.insert_one(dato)
        registros_exitosos += 1
        
    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
Fallo en la arquitectura de scraping: Message: 

CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.
RESUMEN DE CARGA:
Registros exitosos: 0
Registros fallidos: 0


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time  # para capturar fecha y hora

NOMBRE_GRUPO = "Los AveMayo"

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES ---
datos_finales = []
limite_paginas = 3

try:
    driver.get(
        "https://www.jumbo.cl/chocolates-galletas-y-snacks/chocolates"
        "?nombre_promo=destacados-chocolates-26092025"
    )

    # Espera inicial + scroll para disparar carga JS
    time.sleep(5)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")

        # 1. Espera al elemento clave del producto
        WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "a.vtex-product-summary-2-x-clearLink")
            )
        )

        # 2. Captura de bloques de primer nivel
        bloques_primer_nivel = driver.find_elements(
            By.CSS_SELECTOR,
            "div.vtex-product-summary-2-x-container"
        )

        print("Productos encontrados:", len(bloques_primer_nivel))

        for bloque in bloques_primer_nivel:
            try:
                # 3. Identificador (MISMA FORMA QUE ENVIASTE)
                dato_identificador = bloque.find_element(
                    By.CSS_SELECTOR,
                    "a.vtex-product-summary-2-x-clearLink"
                ).get_attribute("title")

                # 4. Precio
                dato_valor = bloque.find_element(
                    By.CSS_SELECTOR,
                    "span.vtex-product-price-1-x-sellingPriceValue"
                ).text

                item_extraido = {
                    "identificador": dato_identificador,
                    "valor": dato_valor,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                }

                datos_finales.append(item_extraido)

            except Exception:
                # Si un producto puntual falla, no se cae todo
                continue

        # --- NAVEGACIÓN: PAGINACIÓN ---
        try:
            disparador_siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "button.vtex-search-result-3-x-paginationButton--next"
            )
            driver.execute_script("arguments[0].click();", disparador_siguiente)
            time.sleep(5)
        except Exception:
            print("Fin del árbol de navegación.")
            break

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

# =========================
# CARGA EN MONGODB
# =========================

from pymongo import MongoClient

try:
    client = MongoClient("database", 27017)
    db = client["proyecto_bigdata"]
    coleccion = db["datos_scraping"]
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        valor_sucio = str(dato.get("valor", "0"))
        valor_limpio = (
            valor_sucio.replace("$", "")
            .replace(".", "")
            .replace(",", "")
            .strip()
        )

        dato["valor"] = float(valor_limpio)

        coleccion.insert_one(dato)
        registros_exitosos += 1

    except Exception:
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
Fallo en la arquitectura de scraping: Message: 

CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.
RESUMEN DE CARGA:
Registros exitosos: 0
Registros fallidos: 0


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time  # para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "Los AveMayo"

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 1  # FerMarket no tiene paginación en esta categoría

try:
    driver.get("https://www.fermarket.cl/linea-select-sofar/bebestibles")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")

        # 1. Espera al Contenedor Raíz
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.CLASS_NAME, "product-item")
            )
        )

        # 2. Captura de Bloques de Primer Nivel
        bloques_primer_nivel = driver.find_elements(
            By.CLASS_NAME, "product-item"
        )

        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (MISMA FORMA)
            dato_identificador = bloque.find_element(
                By.TAG_NAME, "a"
            ).get_attribute("title")

            dato_valor = bloque.find_element(
                By.CLASS_NAME, "price"
            ).text

            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }

            datos_finales.append(item_extraido)

        print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

# =========================
# INSERCIÓN EN MONGODB
# =========================
from pymongo import MongoClient

try:
    client = MongoClient("database", 27017)
    db = client["proyecto_bigdata"]
    coleccion = db["datos_scraping"]
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        valor_sucio = str(dato.get("valor", "0"))
        valor_limpio = (
            valor_sucio.replace("$", "")
            .replace(".", "")
            .strip()
        )

        dato["valor"] = float(valor_limpio)

        coleccion.insert_one(dato)
        registros_exitosos += 1

    except Exception:
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
Fallo en la arquitectura de scraping: Message: 

CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.
RESUMEN DE CARGA:
Registros exitosos: 0
Registros fallidos: 0


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.google.com")
print(driver.title)

driver.quit()

Google


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time  # para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "Los AveMayo"  # Nombre de grupo

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 1  # FerMarket no usa paginación clásica

try:
    driver.get("https://www.fermarket.cl/")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")

        # 1. Espera al Contenedor Raíz
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located(
                (By.CLASS_NAME, "product-item")
            )
        )

        # 2. Captura de Bloques de Primer Nivel (cada producto)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "product-item")

        print("Productos encontrados:", len(bloques_primer_nivel))

        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (MISMA FORMA QUE ENVIASTE)
            dato_identificador = bloque.find_element(By.TAG_NAME, "a").get_attribute("title")
            dato_valor = bloque.find_element(By.CLASS_NAME, "price").text

            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }
            datos_finales.append(item_extraido)

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

# TENEMOS EL DICCIONARIO CON UN FORMATO JSON
# AHORA LO INCLUIMOS EN LA BASE DE DATOS

from pymongo import MongoClient

# 1. Configuración de conexión
try:
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    print("CONEXIÓN ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXIÓN:", e)

# 2. Procesamiento e inserción de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza de símbolos y conversión a float
        valor_sucio = str(dato.get('valor', '0'))
        valor_limpio = valor_sucio.replace('$', '').replace('.', '').strip()

        dato['valor'] = float(valor_limpio)

        # Inserción individual
        coleccion.insert_one(dato)
        registros_exitosos += 1

    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCIÓN:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
Fallo en la arquitectura de scraping: Message: 

CONEXIÓN ESTABLECIDA: Conectado a MongoDB exitosamente.
RESUMEN DE CARGA:
Registros exitosos: 0
Registros fallidos: 0


In [8]:
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.firefox.service import Service
import time

NOMBRE_GRUPO = "Los AveMayo"

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless")

service = Service(GeckoDriverManager().install())
driver = webdriver.Firefox(service=service, options=options)

# --- PASO 2: MEMORIA ---
datos_finales = []

try:
    driver.get("https://www.fermarket.cl/linea-select-sofar/bebestibles")

    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "product-item")
        )
    )

    bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "product-item")
    print("Productos encontrados:", len(bloques_primer_nivel))

    for bloque in bloques_primer_nivel:
        identificador = bloque.find_element(By.TAG_NAME, "a").get_attribute("title")
        valor = bloque.find_element(By.CLASS_NAME, "price").text

        datos_finales.append({
            "identificador": identificador,
            "valor": valor,
            "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
            "grupo": NOMBRE_GRUPO
        })

except Exception as e:
    print("ERROR:", e)

finally:
    driver.quit()

print("Total extraído:", len(datos_finales))

SessionNotCreatedException: Message: Expected browser binary location, but unable to find binary in default location, no 'moz:firefoxOptions.binary' capability provided, and no binary flag set on the command line; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception


In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "Los AveMayo" #Agegaremos el nombre de grupo dentro del diccionario y la hora en que se obtubo

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("http://books.toscrape.com/")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz (el que agrupa todos los resultados)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # 2. Captura de Bloques de Primer Nivel (la "tarjeta" o "card" del item)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "product_pod")
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (Selectores específicos)
            # Extraemos el atributo 'title' del enlace para evitar texto truncado
            dato_identificador = bloque.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
            dato_valor = bloque.find_element(By.CLASS_NAME, "price_color").text
            
            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN: Lógica de Salto de Nodo (Paginación) ---
        try:
            # Buscamos el activador del siguiente nodo de datos
            disparador_siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
            disparador_siguiente.click()
        except Exception:
            print("Fin del árbol de navegación o nodo no encontrado.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

#TENEMOS EL DICCIONARIO CON UN FORMATO JSON, OBTENIDO DEL SCRIPT. AHORA LO INCLUIMOS EN LA BASE DE DATOS
#Cambios en los datos de salida
from pymongo import MongoClient



# 1. Configuracion de conexion
try:
    # 'database' es el nombre del servicio en el docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Procesamiento e insercion de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza de simbolos y conversion a float
        # Se eliminan simbolos de moneda, comas y espacios en blanco
        valor_sucio = str(dato.get('valor', '0'))
        valor_limpio = valor_sucio.replace('£', '').replace(',', '').strip()
        
        dato['valor'] = float(valor_limpio)
        
        # Insercion individual
        coleccion.insert_one(dato)
        registros_exitosos += 1
        
    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos) que cosas en este codigo debo cambiar para poder scrappear

SyntaxError: invalid syntax (1120962634.py, line 110)